# mT5-base — inference, decode sweep (X7), RAG-at-inference (X6)

Run **after** training. Loads one checkpoint and:
1. asserts the parameter cap and that logits are finite,
2. **X6** — feeds RAG-formatted inputs to a plain-trained checkpoint (does RAG need training in?),
3. **X7** — sweeps beams × length_penalty, then **re-verifies the winner on a disjoint dev slice**,
4. decodes the 1,000 competition test rows to `submission.csv`.

🔴 **A well-formed CSV of garbage is indistinguishable from a good one until the leaderboard says
so.** This notebook asserts a known-good dev number *before* writing anything — that assert is the
only thing that has ever caught a silent decode failure in this project.

In [ ]:
# 1 ── config
CKPT      = "runs/X2/best"        # the arm you are decoding
DATA      = "../data/plain"       # ../data/rag for an X3 checkpoint, or for X6
DEV_ROWS  = 300                   # selection slice
VERIFY    = (300, 600)            # 🔴 disjoint slice — nothing was selected on these rows
EVAL_BATCH = 8
MAX_SRC, MAX_NEW = 1024, 512
EXPECTED_DEV_F1 = None            # 🔴 SET THIS from the arm's run.json before decoding test

In [ ]:
# 2 ── env + load
!pip install -q --upgrade "transformers==4.57.3" accelerate sentencepiece
import torch, sys, json, re
import pandas as pd
from pathlib import Path
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
sys.path.insert(0, "../shared")
from evaluate import score_predictions, report

cap  = torch.cuda.get_device_capability()
DTYPE = torch.bfloat16 if cap[0] >= 8 else torch.float32     # 🔴 never fp16
tok   = AutoTokenizer.from_pretrained(CKPT)

model = AutoModelForSeq2SeqLM.from_pretrained(CKPT, dtype=DTYPE).cuda().eval()

n = sum(p.numel() for p in model.parameters())
print(f"parameters: {n:,}")
assert n <= 3_000_000_000, f"3B CAP BREACHED: {n:,}"
TOTAL = n + 247_577_856 + 278_000_000     # + champion + retriever
print(f"✅ specialist within cap. Full pipeline: {TOTAL:,} ({TOTAL/1e9:.2f}B)")
assert TOTAL <= 3_000_000_000, f"🔴 PIPELINE OVER 3B: {TOTAL:,} — drop the retriever or shrink"

with torch.no_grad():   # 🔴 T5 in the wrong dtype emits NaN and still "decodes" cleanly
    probe = tok("হেলো", return_tensors="pt").to("cuda")
    lg = model(**probe, decoder_input_ids=torch.zeros((1,1),dtype=torch.long,device="cuda")).logits
assert torch.isfinite(lg).all(), "❌ NON-FINITE LOGITS — wrong dtype"
print("✅ logits finite")

In [ ]:
# 3 ── decoder
@torch.no_grad()
def generate(texts, num_beams=4, length_penalty=1.0, max_new=MAX_NEW):
    out = []
    for i in range(0, len(texts), EVAL_BATCH):
        enc = tok([str(t) for t in texts[i:i+EVAL_BATCH]], return_tensors="pt",
                  padding=True, truncation=True, max_length=MAX_SRC).to("cuda")
        g = model.generate(**enc, num_beams=num_beams, max_new_tokens=max_new,
                           min_new_tokens=0, length_penalty=length_penalty, do_sample=False)
        out += tok.batch_decode(g, skip_special_tokens=True)
    return out

In [ ]:
# 4 ── X6: does RAG need to be trained in, or can it be bolted on?
#     Point CKPT at an X2 (plain-trained) checkpoint and DATA at ../data/rag, then run this.
RUN_X6 = False
if RUN_X6:
    rag_dev = pd.read_parquet("../data/rag/dev.parquet").iloc[:DEV_ROWS]
    p = generate(rag_dev["input"].tolist())
    r = score_predictions(p, rag_dev["output"].tolist(),
                          ref_examples=rag_dev["ref_output"].tolist())
    report(r, arm="X6 RAG-at-inference-only", model=CKPT)
    print("compare against: X2 (same ckpt, plain input) and X3 (RAG trained in).")
    print("≈X3 -> RAG needs no training.  ≪X3 -> it must be trained in.  ≪X2 -> it actively hurts.")

In [ ]:
# 5 ── X7: decode sweep on the selection slice
dv = pd.read_parquet(f"{DATA}/dev.parquet")
sel = dv.iloc[:DEV_ROWS]
refs = sel["output"].tolist()
ref_ex = sel["ref_output"].tolist() if "ref_output" in sel else None

rows = []
for beams in (4, 8):
    for lp in (1.0, 1.2, 1.5):
        r = score_predictions(generate(sel["input"].tolist(), beams, lp), refs, ref_examples=ref_ex)
        rows.append({"beams": beams, "lp": lp, "token_f1": round(r["token_f1"], 4),
                     "rouge_l": round(r["rouge_l"], 4),
                     "tokens": round(r["mean_pred_tokens"], 1)})
        print(rows[-1], flush=True)
sweep = pd.DataFrame(rows).sort_values("token_f1", ascending=False)
print("\n", sweep.to_string(index=False))
BEST_BEAMS, BEST_LP = int(sweep.iloc[0]["beams"]), float(sweep.iloc[0]["lp"])
print(f"\nwinner: beams={BEST_BEAMS} lp={BEST_LP}")

In [ ]:
# 6 ── 🔴 verify the winner on the DISJOINT slice. A 6-config sweep over 300 rows finds
#      spurious winners; this is how the champion's own decoder was confirmed before shipping.
ver = dv.iloc[VERIFY[0]:VERIFY[1]]
if len(ver):
    rv = score_predictions(generate(ver["input"].tolist(), BEST_BEAMS, BEST_LP),
                           ver["output"].tolist(),
                           ref_examples=ver["ref_output"].tolist() if "ref_output" in ver else None)
    report(rv, arm=f"X7 verify beams={BEST_BEAMS} lp={BEST_LP}", model=CKPT)
    print("A winner that holds up here is real. One that collapses was sweep noise.")
else:
    print("⚠️ dev split has no rows in the verify range — rebuild dev with more rows to verify.")

In [ ]:
# 7 ── 🔴 KNOWN-GOOD GATE, then decode test
assert EXPECTED_DEV_F1 is not None, "🔴 set EXPECTED_DEV_F1 from the arm's run.json first"
got = score_predictions(generate(sel["input"].tolist(), BEST_BEAMS, BEST_LP), refs)["token_f1"]
print(f"dev[0:{DEV_ROWS}] Token F1 = {got:.4f}   (recorded {EXPECTED_DEV_F1:.4f})")
assert got > EXPECTED_DEV_F1 - 0.02, (
    f"❌ {got:.4f} is far below the recorded {EXPECTED_DEV_F1:.4f} — DO NOT write a submission. "
    "Something about this load/decode differs from training.")
print("✅ matches the recorded number — safe to decode test")

test = pd.read_parquet(f"{DATA}/test.parquet")
preds = generate(test["input"].tolist(), BEST_BEAMS, BEST_LP)
sub = pd.DataFrame({"id": test["id"], "output": preds})
assert len(sub) == 1000 and list(sub.columns) == ["id", "output"]
assert not sub["id"].duplicated().any(), "duplicate ids"
assert not (sub["output"].str.strip() == "").any(), "empty predictions"
sub.to_csv("submission.csv", index=False)
print(sub.shape); sub.head(3)

## Deliverables

- `submission.csv` — 1,000 rows, `id,output`
- the decode sweep table + the **disjoint verification** number (not just the selection number)
- winning `beams` / `length_penalty`, recorded in `RESULTS.md`
- X6's verdict if you ran it

🔴 **This notebook covers the Phase 2 specialist branch only.** The full submission also needs the
champion's branch for ChatDoctor-resolving ids — see `../../FINAL SUBMISSION_DRAFT/`. The combined
routing script is still to be written, and needs both halves finalised first.